# Sinmate: QLoRA Fine-Tuning Gemma-3-1B-IT for Spoken Sinhala

This Google Colab notebook fine-tunes **`google/gemma-3-1b-it`** using **4-bit QLoRA** on a conversational Sinhala dataset (*Kathitha Bhashawa* with Sri Lankan fillers, Singlish, and 4 personas).

### Pipeline Overview:
1. **Load Base Model**: `google/gemma-3-1b-it` quantized to 4-bit (VRAM footprint ~1.8GB on free Colab T4).
2. **Chat Template**: Standard Gemma turn tags `<start_of_turn>user...<end_of_turn>`.
3. **Train with QLoRA**: LoRA adapters on linear attention and MLP layers (`r=16, alpha=32`).
4. **Inference Test**: Multi-persona conversational test in colloquial Sinhala.
5. **Export GGUF**: Export 4-bit quantized GGUF weights (~700MB–800MB) for offline laptop execution.

In [1]:
# 1. Install fine-tuning dependencies
!pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece

In [2]:
# 2. Hugging Face Login (Access Gemma license)
from google.colab import userdata
from huggingface_hub import login

try:
    token = userdata.get('HF_TOKEN')
    login(token=token)
except Exception:
    print("Enter your Hugging Face access token:")
    login()

Enter your Hugging Face access token:


In [ ]:
# 3. Clone or upload Sinmate dataset
import os
if not os.path.exists("data"):
    !git clone https://github.com/chathuwa-whiz/Sinmate.git
    %cd Sinmate

from datasets import load_dataset
dataset = load_dataset("json", data_files={"train": "data/train.jsonl", "val": "data/val.jsonl"})
print("Dataset loaded:", dataset)

In [ ]:
# 4. Load Gemma-3-1B-IT in 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-3-1b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 5. Format Chat Templates & Train
from trl import SFTTrainer
from transformers import TrainingArguments

def format_prompts(example):
    formatted = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": formatted}

formatted_dataset = dataset.map(format_prompts)

training_args = TrainingArguments(
    output_dir="./gemma3_sinhala_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    num_train_epochs=3,
    fp16=False,
    bf16=True,
    save_strategy="epoch",
    evaluation_strategy="epoch"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["val"],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()

In [ ]:
# 6. Test Interactive Multi-Persona Inference
test_messages = [
    {"role": "system", "content": "You are Sinmate, an affectionate native Sinhala companion.\n[ROLE: BEST_FRIEND]\nUse friendly colloquial Sinhala (machan), strictly 1-2 spoken sentences."},
    {"role": "user", "content": "Mata ada mara kammali ban, monada karanna hoda?"}
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.7, top_p=0.9)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n🤖 SINMATE RESPONSE:", response)

In [ ]:
# 7. Save LoRA Adapters
model.save_pretrained("sinmate-gemma-3-1b-lora")
tokenizer.save_pretrained("sinmate-gemma-3-1b-lora")
print("Adapters saved successfully to sinmate-gemma-3-1b-lora!")